# Scenario: Finding the Bottleneck in Emergency Alerts

In [3]:
import pandas as pd
import sqlite3
# creating dataset representing AI processing timestamps (Format: YYYY-MM-DD HH:MM:SS)
latency_data = {
    "scan_id": [9001, 9002, 9003, 9004, 9005],
    "patient_id": ["P-701", "P-702", "P-703", "P-704", "P-705"],
    "scan_completed_time": ["2026-05-31 14:00:00", "2026-05-31 14:15:00", "2026-05-31 14:30:00", "2026-05-31 15:00:00", "2026-05-31 15:10:00"],
    "ai_alert_sent_time": ["2026-05-31 14:01:30", "2026-05-31 14:22:15", "2026-05-31 14:31:10", "2026-05-31 15:12:45", "2026-05-31 15:11:15"]
}
# adding dataset into DataFrame
df_latency_data = pd.DataFrame(latency_data)
# creating sql and save DataFrame in the temp memory
connt = sqlite3.connect(":memory:")
df_latency_data.to_sql("ai_performance_logs", connt, index = False, if_exists = "replace")
# function to run the query
def run_query(query):
    return pd.read_sql_query(query, connt)
print("***************************************** Latency Audit Database is ready! ****************")

***************************************** Latency Audit Database is ready! ****************


# Calculating Delay in Minutes

In [19]:
# all data to review
all_data = "SELECT * FROM ai_performance_logs"
print("********************************* all data to review ******************")
print()
display(run_query(all_data))
# SQL query that calculates the exact difference in minutes between scan_completed_time and ai_alert_sent_time for every scan.
alert_gap = """
SELECT
    scan_id,
    scan_completed_time,
    ai_alert_sent_time,
    ROUND((CAST(strftime('%s', ai_alert_sent_time) AS REAL) - CAST(strftime('%s', scan_completed_time) AS REAL)) / 60, 2) AS delay_minutes
FROM ai_performance_logs
"""
print("******************** time difference between scan_completed_time and ai_alert_sent_time ************ ")
display(run_query(alert_gap))

********************************* all data to review ******************



,scan_id,patient_id,scan_completed_time,ai_alert_sent_time
0,9001,P-701,2026-05-31 14:00:00,2026-05-31 14:01:30
1,9002,P-702,2026-05-31 14:15:00,2026-05-31 14:22:15
2,9003,P-703,2026-05-31 14:30:00,2026-05-31 14:31:10
3,9004,P-704,2026-05-31 15:00:00,2026-05-31 15:12:45
4,9005,P-705,2026-05-31 15:10:00,2026-05-31 15:11:15


******************** time difference between scan_completed_time and ai_alert_sent_time ************ 


,scan_id,scan_completed_time,ai_alert_sent_time,delay_minutes
0,9001,2026-05-31 14:00:00,2026-05-31 14:01:30,1.50
1,9002,2026-05-31 14:15:00,2026-05-31 14:22:15,7.25
2,9003,2026-05-31 14:30:00,2026-05-31 14:31:10,1.17
3,9004,2026-05-31 15:00:00,2026-05-31 15:12:45,12.75
4,9005,2026-05-31 15:10:00,2026-05-31 15:11:15,1.25


# Isolating Regulatory Violations

In [20]:
# SQL query to filter and display only the records where the processing delay was greater than 3.0 minutes.
delayed_reports = """
SELECT
    scan_id,
    scan_completed_time,
    ai_alert_sent_time,
    ROUND((CAST(strftime('%s', ai_alert_sent_time) AS REAL) - CAST(strftime('%s', scan_completed_time) AS REAL)) / 60, 2) AS delay_minutes
FROM ai_performance_logs
WHERE delay_minutes > 3.0;
"""
print("******************************* delayed reports ***************")
display(run_query(delayed_reports))


******************************* delayed reports ***************


,scan_id,scan_completed_time,ai_alert_sent_time,delay_minutes
0,9002,2026-05-31 14:15:00,2026-05-31 14:22:15,7.25
1,9004,2026-05-31 15:00:00,2026-05-31 15:12:45,12.75
